# Synthetic Cohort V2 Method Note

This notebook documents the shift from the original synthetic cohort generator to the separate latent-factor `v2` generator introduced for evaluation. It focuses on four questions:

1. What the old logic did.
2. What the new logic does instead.
3. What changed structurally in the generated patients.
4. How the evaluation results changed after moving to the new cohort.


## Old Logic

The original cohort generator was useful for early smoke tests, but it had a few properties that made the evaluation unrealistically easy or unstable:

- Profiles were generated close to hand-authored disease centroids.
- Borderline cases were often softened positives rather than truly ambiguous patients.
- Negative cases and mimics were too weak, so competing conditions did not meaningfully overlap.
- Bayesian clarification answers were emitted too deterministically from the target condition.
- Missingness and contradictions were limited, so the synthetic profiles looked cleaner than real user flows.

This created two visible artifacts:

- Some diseases became unrealistically separable, producing suspiciously strong recall.
- Some clinically sensible Bayesian questions showed near-zero information gain because the generator had already baked the answer into the profile too directly.


## New Logic

The `v2` generator in [cohort_generator_v2_latent.py](/Users/annaesakova/aipm/halfFull/evals/cohort_generator_v2_latent.py) keeps the original generator untouched and creates a separate cohort at [profiles_v2_latent.json](/Users/annaesakova/aipm/halfFull/evals/cohort/profiles_v2_latent.json).

The new logic is built around latent factors rather than direct disease templates:

- First sample latent patient state: physiology, symptom burden, menstrual status, sleep burden, inflammatory burden, hydration, bleeding risk, alcohol exposure, and related drivers.
- Then emit observed data from that latent state: quiz answers, Bayesian clarification answers, and labs.
- Inject noise so observed answers are not deterministic reflections of the target disease.
- Add mimic and overlap cases deliberately so more users sit in the realistic confusion zones between conditions.
- Preserve healthy controls and edge cases as explicit cohort slices.

This makes the cohort harder in a more realistic way: not by random corruption, but by increasing clinically plausible overlap.


## What Changed

Compared with the original generator, the `v2` latent generator changes the cohort in these ways:

- Disease signal is no longer encoded as a near-direct symptom bundle.
- Borderline users are generated as genuinely ambiguous patients rather than diluted positives.
- Competing explanations are stronger, especially for fatigue-like presentations.
- Bayesian answers are now driven by noisy latent state, which makes clarification questions more meaningful.
- Overlap between conditions is intentional, so top-k ranking and posterior updates are tested under more realistic pressure.

The generated `v2` cohort contains:

- 600 profiles
- 11 conditions
- 30 healthy controls
- 20 edge cases
- 578 profiles with labs


## How The Evals Went

### ML-only comparison

| Metric | Old cohort | New latent cohort |
|---|---:|---:|
| Top-1 accuracy | 16.6% | 17.6% |
| Top-3 coverage | 37.8% | 42.2% |
| Healthy over-alert rate | 93.3% | 90.0% |

Interpretation:

- The ML layer is still weak on both cohorts.
- The new cohort is not simply harsher everywhere; it is more overlapped and more structured.
- The models still over-flag heavily, which is a useful signal rather than a generator failure.

Full ML report:

- Old cohort: [layer1_20260325_222624.md](/Users/annaesakova/aipm/halfFull/evals/reports/layer1_20260325_222624.md)
- New latent cohort: [layer1_20260325_231306.md](/Users/annaesakova/aipm/halfFull/evals/reports/layer1_20260325_231306.md)

### Bayesian comparison

| Metric | Old cohort | New latent cohort |
|---|---:|---:|
| Pre-update Brier | 0.2221 | 0.1570 |
| Post-update Brier | 0.2191 | 0.1124 |
| Brier delta | -0.0030 | -0.0446 |
| Top-5 coverage delta | +3.13 pp | +7.01 pp |
| Low-gain questions flagged | 24 | 13 |

Interpretation:

- On the original cohort, Bayesian improvement existed but was small.
- On the latent `v2` cohort, Bayesian updating provides a clearer and more believable lift.
- The drop in low-gain questions suggests the new cohort creates better clarification scenarios.

Full Bayesian report:

- Old cohort: [bayesian_20260325_221003.md](/Users/annaesakova/aipm/halfFull/evals/reports/bayesian_20260325_221003.md)
- New latent cohort: [bayesian_20260325_231751.md](/Users/annaesakova/aipm/halfFull/evals/reports/bayesian_20260325_231751.md)


## Takeaways

- The original cohort likely overstated separability for several diseases.
- The latent `v2` cohort produces a more credible evaluation environment for both ranking and Bayesian clarification.
- The base ML layer remains the main weakness.
- The Bayesian layer looks more valuable under the new cohort because it is resolving realistic ambiguity rather than polishing already-obvious cases.

Recommended next use of `v2`:

- keep it as a separate benchmark, not a replacement yet
- use it to tune question sets and disease overlap assumptions
- use the old and new cohorts together to detect both regressions and overfitting to a single synthetic style
